In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8cd319b4f2187b6b29bb69603a96460fc325a975"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8cd319b4f2187b6b29bb69603a96460fc325a975/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Utility Functions and Classes


This section contains the implementations of utility functions and classes used in this book.

In [1]:
import inspect
from IPython import display
import collections
from d2l import jax as d2l
import jax

Hyperparameters.

In [2]:
@d2l.add_to_class(d2l.HyperParameters)
def save_hyperparameters(self, ignore=None):
    """Save function arguments into class attributes."""
    ignore = [] if ignore is None else ignore
    frame = inspect.currentframe().f_back
    _, _, _, local_vars = inspect.getargvalues(frame)
    self.hparams = {k:v for k, v in local_vars.items()
                    if k not in set(ignore+['self']) and not k.startswith('_')}
    for k, v in self.hparams.items():
        setattr(self, k, v)

Progress bar.

In [3]:

import io, os, queue, threading, time

@d2l.add_to_class(d2l.ProgressBoard)
def draw(self, x, y, label, every_n=1):
    """Schedule the point (x, y) to be plotted under `label`.

    This call is *asynchronous*: it appends the point to an internal queue
    and returns immediately, so the (possibly compiled) training loop never
    blocks on a device-to-host transfer or on matplotlib. `y` may be a number
    or a zero-argument callable returning one; if it is a callable, the
    conversion runs on the background drawing thread, not on the caller's.
    `every_n` averages each group of `n` raw points into one plotted point."""
    self._start_drawer()
    try:
        self._queue.put_nowait((x, y, label, every_n))
    except queue.Full:
        pass  # drop the point rather than stall the training loop

@d2l.add_to_class(d2l.ProgressBoard)
def _start_drawer(self):
    if getattr(self, '_drawer', None) is not None:
        return
    self._queue = queue.Queue(maxsize=1000)
    self._lock = threading.Lock()
    self._handle = None
    self._last = 0.0
    self._running = True
    # Emit live frames only outside the headless book build, so an executed
    # notebook records exactly one (final) figure in the committed store.
    self._live = self.display and os.environ.get('D2L_NB_CAPTURE') != '1'
    self._drawer = threading.Thread(target=self._drain, daemon=True)
    self._drawer.start()

@d2l.add_to_class(d2l.ProgressBoard)
def _drain(self):
    while self._running or not self._queue.empty():
        try:
            batch = [self._queue.get(timeout=0.1)]
        except queue.Empty:
            continue
        while True:  # coalesce everything currently queued
            try: batch.append(self._queue.get_nowait())
            except queue.Empty: break
        for x, y, label, every_n in batch:
            self._accumulate(x, y() if callable(y) else y, label, every_n)
        if self._live and time.time() - self._last > 0.1:
            self._render(thread=True)
            self._last = time.time()

@d2l.add_to_class(d2l.ProgressBoard)
def _accumulate(self, x, y, label, every_n):
    Point = collections.namedtuple('Point', ['x', 'y'])
    with self._lock:
        if not hasattr(self, 'raw_points'):
            self.raw_points = collections.OrderedDict()
            self.data = collections.OrderedDict()
        if label not in self.raw_points:
            self.raw_points[label] = []
            self.data[label] = []
        points = self.raw_points[label]
        points.append(Point(x, float(y)))
        if len(points) != every_n:
            return
        mean = lambda v: sum(v) / len(v)
        self.data[label].append(Point(mean([p.x for p in points]),
                                      mean([p.y for p in points])))
        points.clear()

@d2l.add_to_class(d2l.ProgressBoard)
def _plot_lines(self, axes):
    with self._lock:
        series = [(k, list(v)) for k, v in getattr(self, 'data', {}).items()]
    for (k, v), ls, color in zip(series, self.ls, self.colors):
        axes.plot([p.x for p in v], [p.y for p in v],
                  linestyle=ls, color=color, label=k)
    if self.xlim: axes.set_xlim(self.xlim)
    if self.ylim: axes.set_ylim(self.ylim)
    if not self.xlabel: self.xlabel = self.x
    axes.set_xlabel(self.xlabel)
    axes.set_ylabel(self.ylabel)
    axes.set_xscale(self.xscale)
    # A smoothed series exists before it necessarily contains its first point.
    # Matplotlib rejects an empty/nonpositive log axis, so keep the temporary
    # live frame linear and apply the requested log scale as soon as a positive
    # value is available. The final training figure therefore still uses log.
    has_positive_y = any(p.y > 0 for _, values in series for p in values)
    if self.yscale != 'log' or has_positive_y:
        axes.set_yscale(self.yscale)
    if series: axes.legend()

@d2l.add_to_class(d2l.ProgressBoard)
def _render(self, thread=False):
    if not self.display:
        return
    if thread:
        # Off the main thread: matplotlib's global pyplot state is not
        # thread-safe, so render into a private Agg figure and push a PNG.
        from matplotlib.figure import Figure
        from matplotlib.backends.backend_agg import FigureCanvasAgg
        fig = Figure(figsize=self.figsize)
        FigureCanvasAgg(fig)
        self._plot_lines(fig.add_subplot(111))
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        frame = display.Image(data=buf.getvalue())
        if self._handle is None:
            self._handle = display.display(frame, display_id=True)
        else:
            self._handle.update(frame)
    else:
        # Main thread (final frame): render through pyplot so the output is
        # captured exactly like the rest of the book's inline figures.
        d2l.use_svg_display()
        if self.fig is None:
            self.fig = d2l.plt.figure(figsize=self.figsize)
        self.fig.clf()
        self._plot_lines(self.fig.add_subplot(111))
        if self._handle is not None:
            self._handle.update(self.fig)         # interactive: reuse live slot
        else:
            # Display, then a trailing clear_output(wait=True): the inline
            # backend's automatic end-of-cell render then *replaces* (rather
            # than duplicates) this figure, so exactly one image is captured.
            display.display(self.fig)
            display.clear_output(wait=True)

@d2l.add_to_class(d2l.ProgressBoard)
def flush(self):
    """Wait for every scheduled point to be drawn, then render the final
    figure on the calling thread. `Trainer.fit` calls this for you; call it
    yourself after a bare `draw` loop."""
    if getattr(self, '_drawer', None) is not None:
        self._running = False
        self._drawer.join()
        self._drawer = None
    self._render(thread=False)

Add FrozenLake environment

Create environment

Show value function

Show Q function

Trainer

Legacy helper functions retained for backward compatibility:

In [4]:

def linreg(X, w, b):
    """The linear regression model."""
    return d2l.matmul(X, w) + b

def squared_loss(y_hat, y):
    """Squared loss."""
    return (y_hat - d2l.reshape(y, y_hat.shape)) ** 2 / 2

def load_array(data_arrays, batch_size, is_train=True):
    """Construct a JAX-compatible data iterator.

    For training, the last partial batch is dropped so every minibatch
    has the same shape — this lets a `@jax.jit`'d step function compile
    once per epoch instead of recompiling for the smaller last batch.
    Validation iterators yield all batches (the last may be smaller),
    matching how PyTorch / TF / MXNet behave.
    """
    # Keep dataset storage on the host. Only minibatches need to be transferred
    # to an accelerator, and host-side indexing avoids launching a device gather
    # for every array in every minibatch.
    data_arrays = tuple(np.asarray(a) for a in data_arrays)
    n = data_arrays[0].shape[0]
    indices = np.arange(n)
    last = n - (n % batch_size) if is_train else n
    def data_iter():
        if is_train:
            np.random.shuffle(indices)
            # Shuffle each full field once, then transfer contiguous slices.
            # This avoids a separate fancy-index gather for every field and
            # every minibatch.
            epoch_arrays = tuple(a[indices] for a in data_arrays)
        else:
            epoch_arrays = data_arrays
        for i in range(0, last, batch_size):
            yield tuple(jnp.array(a[i: min(i + batch_size, n)])
                        for a in epoch_arrays)
    class DataIter:
        def __iter__(self):
            return data_iter()
        def __len__(self):
            return last // batch_size if is_train else (n + batch_size - 1) // batch_size
    return DataIter()

class ArrayDataLoader:
    """A simple data loader for JAX that batches arrays or dataset objects."""
    def __init__(self, *args, batch_size=32, shuffle=False, drop_last=False,
                 **kwargs):
        if len(args) == 1 and hasattr(args[0], '__len__') and hasattr(args[0], '__getitem__'):
            dataset = args[0]
            items = [dataset[i] for i in range(len(dataset))]
            if isinstance(items[0], (tuple, list)):
                self.arrays = tuple(np.array([item[j] for item in items])
                                    for j in range(len(items[0])))
            else:
                self.arrays = (np.array(items),)
        else:
            self.arrays = tuple(np.asarray(a) for a in args)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last

    def __iter__(self):
        n = self.arrays[0].shape[0]
        indices = np.arange(n)
        if self.shuffle:
            np.random.shuffle(indices)
        for i in range(0, n, self.batch_size):
            end = i + self.batch_size
            if self.drop_last and end > n:
                break
            batch_idx = indices[i:min(end, n)]
            yield tuple(jnp.array(a[batch_idx]) for a in self.arrays)

    def __len__(self):
        n = self.arrays[0].shape[0]
        if self.drop_last:
            return n // self.batch_size
        return (n + self.batch_size - 1) // self.batch_size

class Animator:
    """For plotting data in animation."""
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        if legend is None:
            legend = []
        d2l.use_svg_display()
        self.fig, self.axes = d2l.plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        self.config_axes = lambda: d2l.set_axes(
            self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)

class Accumulator:
    """For accumulating sums over `n` variables."""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def tokenize(lines, token='word'):
    """Split text lines into word or character tokens."""
    assert token in ('word', 'char'), 'Unknown token type: ' + token
    return [line.split() if token == 'word' else list(line) for line in lines]

def truncate_pad(line, num_steps, padding_token):
    """Truncate or pad sequences."""
    if len(line) > num_steps:
        return line[:num_steps]
    return line + [padding_token] * (num_steps - len(line))

def download_extract(name, folder=None):
    """Download and extract a zip/tar file."""
    fname = download(name)
    base_dir = os.path.dirname(fname)
    data_dir, ext = os.path.splitext(fname)
    target = os.path.join(base_dir, folder) if folder else data_dir
    marker = fname + '.extracted'
    if os.path.exists(marker):
        return target
    if ext == '.zip':
        fp = zipfile.ZipFile(fname, 'r')
    elif ext in ('.tar', '.gz'):
        fp = tarfile.open(fname, 'r')
    else:
        assert False, 'Only zip/tar files can be extracted.'
    fp.extractall(base_dir)
    open(marker, 'w').close()
    return target

def evaluate_loss(net, data_iter, loss):
    """Evaluate the loss of a model on the given dataset."""
    metric = d2l.Accumulator(2)
    for X, y in data_iter:
        out = net(X)
        y = d2l.reshape(y, out.shape)
        l = loss(out, y)
        metric.add(d2l.reduce_sum(l), d2l.size(l))
    return metric[0] / metric[1]

In [5]:
def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5):
    """Plot a list of images."""
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        try:
            img = d2l.numpy(img)
        except:
            pass
        ax.imshow(img)
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i])
    return axes

In [6]:

import os
import requests
import zipfile
import tarfile
import hashlib

def download(url, folder='../data', sha1_hash=None):
    """Download a file to folder and return the local filepath."""
    if not url.startswith('http'):
        # For back compatibility
        url, sha1_hash = DATA_HUB[url]
    os.makedirs(folder, exist_ok=True)
    fname = os.path.join(folder, url.split('/')[-1])
    # Check if hit cache
    if os.path.exists(fname) and sha1_hash:
        sha1 = hashlib.sha1()
        with open(fname, 'rb') as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname
    # Download
    import tempfile
    print(f'Downloading {fname} from {url}...')
    r = requests.get(url, stream=True, verify=True)
    fd, tmp = tempfile.mkstemp(dir=folder)
    try:
        with os.fdopen(fd, 'wb') as f:
            f.write(r.content)
        os.replace(tmp, fname)
    except BaseException:
        os.unlink(tmp)
        raise
    return fname

def extract(filename, folder=None):
    """Extract a zip/tar file into folder."""
    import tempfile, shutil
    base_dir = os.path.dirname(filename)
    _, ext = os.path.splitext(filename)
    assert ext in ('.zip', '.tar', '.gz'), 'Only support zip/tar files.'
    opener = zipfile.ZipFile if ext == '.zip' else tarfile.open
    if folder is None:
        folder = base_dir
    tmp = tempfile.mkdtemp(dir=folder)
    try:
        with opener(filename, 'r') as fp:
            fp.extractall(tmp)
        for name in os.listdir(tmp):
            src = os.path.join(tmp, name)
            dst = os.path.join(folder, name)
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            os.rename(src, dst)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

More for the attention chapter.

## A Note on Framework Coverage

The legacy helpers in this section (`evaluate_accuracy`, `train_ch6`,
`train_seq2seq`, `predict_seq2seq`, `MaskedSoftmaxCELoss`) are kept for
parity with the original D2L implementation, which predates the unified
`d2l.Trainer` class introduced in this edition. They are deliberately
not provided for JAX, and PyTorch only ships the subset that is genuinely
useful outside of the `Trainer` flow. If you are reading the JAX tab,
the corresponding chapters use `d2l.Trainer.fit(model, data)` end-to-end;
the per-batch logic that these helpers spell out lives inside
`Trainer.fit_epoch` and the `@d2l.add_to_class` extensions defined
alongside it. The MXNet and TensorFlow tabs additionally retain
`evaluate_accuracy` because some of their earlier-chapter snippets call
it directly; for PyTorch and JAX,
use `Trainer.validation_step` (or its return value) instead.